<a href="https://colab.research.google.com/github/varunkshatriya/flyrank-ml-internship-starter-clone/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [29]:
import duckdb

# Create DuckDB connection
con = duckdb.connect()

# Enable remote Hugging Face access
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

print("DuckDB connection created successfully.")

DuckDB connection created successfully.


In [30]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN secret could not be loaded.")

print("Hugging Face token loaded successfully.")

Hugging Face token loaded successfully.


In [31]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. "
        "Set it as an environment variable or load it from your notebook secrets."
    )

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [32]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. "
        "Set it as an environment variable or load it from your notebook secrets."
    )

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [33]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. "
        "Set it as an environment variable or load it from your notebook secrets."
    )

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE huggingface,
    TOKEN '{hf_token}'
);
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [34]:
import os

print(os.getcwd())

/content


In [35]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN secret could not be loaded.")

print("HF token loaded successfully.")
print("Token prefix:", hf_token[:6] + "...")

HF token loaded successfully.
Token prefix: hf_zoZ...


In [36]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN ?
)
""", [hf_token])

print("DuckDB and Hugging Face authentication configured.")

DuckDB and Hugging Face authentication configured.


In [37]:
from google.colab import userdata
from huggingface_hub import whoami

hf_token = userdata.get("HF_TOKEN")

user_info = whoami(token=hf_token)
print("Authenticated as:", user_info["name"])

Authenticated as: varunAiagent


In [38]:
import duckdb

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

print("DuckDB and Hugging Face authentication configured.")

DuckDB and Hugging Face authentication configured.


In [39]:
test = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
)
LIMIT 5
""").df()

display(test)

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


### Loading a dataset from Hugging Face Hub with DuckDB

Now that Hugging Face authentication is configured, you can directly query datasets from the Hugging Face Hub using the `hf://` protocol. For example, to load `duckdb/parquet-test`:

In [40]:
con.execute("""
CREATE OR REPLACE VIEW fact_content_daily_performance AS
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
);
""")

print("Warehouse view created successfully.")

Warehouse view created successfully.


In [41]:
print("Acknowledging 'done'. Please run the previous cell (897fe4a7) to confirm secret setup for 'HF_TOKEN'.")
# If cell 897fe4a7 executes without a SecretNotFoundError, the authentication is set up.

Acknowledging 'done'. Please run the previous cell (897fe4a7) to confirm secret setup for 'HF_TOKEN'.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [42]:
from IPython.display import Markdown, display

display(Markdown("""
### Feature
- `gsc_clicks` — past search clicks; used as a signal of current content performance.
- `gsc_impressions` — past search visibility.
- `gsc_ctr` — click-through rate from search.
- `gsc_avg_position` — average search position.
- `ga4_sessions` — traffic engagement signal, only when `ga4_data_available IS TRUE`.

### Label / proxy
- A future content-performance outcome derived from performance after the prediction date.
- For this contract stage, the exact future label is kept separate from the feature columns to avoid leakage.

### Context
- `report_date` — identifies the daily observation and prediction time.
- `client_id` — used for grouping and future client-aware validation.
- `content_id` — identifies the content item.

### Excluded
- `ga4_data_available` — used as an availability filter, not a model feature.
- Any future-window metric — excluded because it would leak information from after the prediction moment.
- Product decision flags or existing scores — excluded because they can encode an existing decision.
- `client_id` and `content_id` — excluded from model features because they are identifiers and can encourage memorization.
"""))


### Feature
- `gsc_clicks` — past search clicks; used as a signal of current content performance.
- `gsc_impressions` — past search visibility.
- `gsc_ctr` — click-through rate from search.
- `gsc_avg_position` — average search position.
- `ga4_sessions` — traffic engagement signal, only when `ga4_data_available IS TRUE`.

### Label / proxy
- A future content-performance outcome derived from performance after the prediction date.
- For this contract stage, the exact future label is kept separate from the feature columns to avoid leakage.

### Context
- `report_date` — identifies the daily observation and prediction time.
- `client_id` — used for grouping and future client-aware validation.
- `content_id` — identifies the content item.

### Excluded
- `ga4_data_available` — used as an availability filter, not a model feature.
- Any future-window metric — excluded because it would leak information from after the prediction moment.
- Product decision flags or existing scores — excluded because they can encode an existing decision.
- `client_id` and `content_id` — excluded from model features because they are identifiers and can encourage memorization.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [43]:
# ============================================================
# QUERY 1 — Verify the grain
# ============================================================

q1 = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM fact_content_daily_performance
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
"""

grain_check = con.execute(q1).df()

print("QUERY 1 — GRAIN CHECK")
display(grain_check)

if len(grain_check) == 0:
    print("Result: No duplicate report_date × client_hash_id × content_hash_id combinations found.")
    print("The claimed grain holds for this March 2026 slice.")


# ============================================================
# QUERY 2 — Row count and date span
# ============================================================

q2 = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS client_count,
    COUNT(DISTINCT content_hash_id) AS content_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM fact_content_daily_performance
WHERE month = '2026-03'
"""

count_window_check = con.execute(q2).df()

print("\\nQUERY 2 — ROW COUNT AND DATE SPAN")
display(count_window_check)


# ============================================================
# QUERY 3 — Availability
# ============================================================

q3 = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS rows_surviving_availability_filter,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS NOT TRUE
    ) AS rows_removed,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS pct_surviving

FROM fact_content_daily_performance
WHERE month = '2026-03'
"""

availability_check = con.execute(q3).df()

print("\\nQUERY 3 — AVAILABILITY CHECK")
display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1 — GRAIN CHECK


,report_date,client_hash_id,content_hash_id,duplicate_rows


Result: No duplicate report_date × client_hash_id × content_hash_id combinations found.
The claimed grain holds for this March 2026 slice.
\nQUERY 2 — ROW COUNT AND DATE SPAN


,row_count,client_count,content_count,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

\nQUERY 3 — AVAILABILITY CHECK


,total_rows,rows_surviving_availability_filter,rows_removed,pct_surviving
0,9841378,413966,9427412,4.21


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [44]:
from IPython.display import Markdown, display

display(Markdown("""
### Named limitation of this slice

This March 2026 slice does not provide equal history for every client. Some clients began
collecting GSC or GA4 data later than others, so missing earlier history does not mean zero
performance.

Rows where `ga4_data_available` is not true contain zero-filled GA4 values that should not be
interpreted as real zero engagement. For analyses using GA4 metrics, I therefore filter with
`ga4_data_available IS TRUE`.

This is also a single-month contract check. Results observed for March 2026 may not represent
all months or all clients equally. Any future label must use a strictly later time window so
that March features do not overlap the outcome window.
"""))


### Named limitation of this slice

This March 2026 slice does not provide equal history for every client. Some clients began
collecting GSC or GA4 data later than others, so missing earlier history does not mean zero
performance.

Rows where `ga4_data_available` is not true contain zero-filled GA4 values that should not be
interpreted as real zero engagement. For analyses using GA4 metrics, I therefore filter with
`ga4_data_available IS TRUE`.

This is also a single-month contract check. Results observed for March 2026 may not represent
all months or all clients equally. Any future label must use a strictly later time window so
that March features do not overlap the outcome window.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.